In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [3]:
movies.sample()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
849,55000000,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 18, ""name...",http://www.changelingmovie.net,3580,"[{""id"": 417, ""name"": ""corruption""}, {""id"": 456...",en,Changeling,Christine Collins is overjoyed when her kidnap...,27.50916,"[{""name"": ""Imagine Entertainment"", ""id"": 23}, ...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2008-01-30,113020255,141.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"To find her son, she did what no one else dared.",Changeling,7.3,1131


In [4]:
credits.sample()

,movie_id,title,cast,crew
1755,63574,Joyful Noise,"[{""cast_id"": 2, ""character"": ""Vi Rose Hill"", ""...","[{""credit_id"": ""569074a09251414561000c1e"", ""de..."


In [5]:
movies = movies.merge(credits, on='title')

In [6]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [7]:
movies = movies[['movie_id', 'keywords', 'title','genres', 'overview', 'cast', 'crew']]

In [8]:
movies.sample()

,movie_id,keywords,title,genres,overview,cast,crew
320,855,"[{""id"": 1968, ""name"": ""prisoners of war""}, {""i...",Black Hawk Down,"[{""id"": 28, ""name"": ""Action""}, {""id"": 36, ""nam...",When U.S. Rangers and an elite Delta Force tea...,"[{""cast_id"": 20, ""character"": ""SSgt. Matt Ever...","[{""credit_id"": ""52fe4282c3a36847f8024871"", ""de..."


### data preprocessing

In [9]:
# handling null values
movies.isnull().sum()

movie_id    0
keywords    0
title       0
genres      0
overview    3
cast        0
crew        0
dtype: int64

In [10]:
# since we have very less null values we can remove them.
movies.dropna(inplace=True)

In [11]:
# check for duplicate values
movies.duplicated().sum()

np.int64(0)

In [12]:
movies.iloc[0]['genres']

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [13]:
#  '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'
#  we want it in a format ['Action', 'Adventure', 'Fantasy', 'Science Fiction']
#  we will use a convertGenre helper function for that

In [14]:
# convert string movies.iloc[0]['genres'] to object
import ast
ast.literal_eval(movies.iloc[0]['genres'])

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [15]:
def convertGenre(object):
    list = []
    for subobj in ast.literal_eval(object):
        list.append(subobj['name'])
    return list

In [16]:
convertGenre(movies.iloc[0]['genres'])

['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [17]:
#  do same for every index in movies
movies['genres'] = movies['genres'].apply(convertGenre)

In [18]:
movies['genres']

0       [Action, Adventure, Fantasy, Science Fiction]
1                        [Adventure, Fantasy, Action]
2                          [Action, Adventure, Crime]
3                    [Action, Crime, Drama, Thriller]
4                [Action, Adventure, Science Fiction]
                            ...                      
4804                        [Action, Crime, Thriller]
4805                                [Comedy, Romance]
4806               [Comedy, Drama, Romance, TV Movie]
4807                                               []
4808                                    [Documentary]
Name: genres, Length: 4806, dtype: object

In [19]:
movies['keywords'] = movies['keywords'].apply(convertGenre)

In [20]:
def convertCast(object):
    list = []
    counter=0;
    for subobj in ast.literal_eval(object):
        if counter>=3: break
        list.append(subobj['name'])
        counter+=1
    return list

In [21]:
movies['cast'] = movies['cast'].apply(convertCast)

In [22]:
def convertCrew(obj):
    list = []
    for subobj in ast.literal_eval(obj):
        if(subobj['job']=='Director'):
            list.append(subobj['name'])
            break
    return list

In [23]:
movies['crew'] = movies['crew'].apply(convertCrew)

In [24]:
movies.iloc[0]['crew']

['James Cameron']

In [25]:
movies.iloc[0]['cast']

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']

In [26]:
# convert overview into array of tags
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [27]:
movies.sample(5)

,movie_id,keywords,title,genres,overview,cast,crew
4159,13516,[presidential elections],Slacker Uprising,[Documentary],"[Slacker, Uprising, is, a, movie, of, Michael,...","[Michael Moore, Eddie Vedder, Robert Ellis Orr...",[Michael Moore]
415,49049,"[usa, corruption, crime fighter, judge, metrop...",Dredd,"[Action, Science Fiction]","[In, the, future,, America, is, a, dystopian, ...","[Karl Urban, Olivia Thirlby, Lena Headey]",[Pete Travis]
2047,1599,"[sex, detective, jealousy, humor, protest, wif...",I Heart Huckabees,"[Comedy, Romance]","[A, husband-and-wife, team, play, detective,, ...","[Jason Schwartzman, Dustin Hoffman, Lily Tomlin]",[David O. Russell]
4181,1705,"[post-apocalyptic, dystopia, ape]",Battle for the Planet of the Apes,"[Action, Science Fiction]","[The, fifth, and, final, episode, in, the, Pla...","[Roddy McDowall, Natalie Trundy, Austin Stoker]",[J. Lee Thompson]
1421,336004,"[casino, robbery, bus hijacking, heist]",Heist,"[Crime, Action, Thriller]","[A, father, is, without, the, means, to, pay, ...","[Jeffrey Dean Morgan, Robert De Niro, Kate Bos...",[Scott Mann]


In [28]:
# remove spaces from same word like names of same person
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(' ', '') for i in x])
movies['overview'] = movies['overview'].apply(lambda x: [i.replace(' ', '') for i in x])

movies['cast'] = movies['cast'].apply(lambda x: [i.replace(' ', '') for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(' ', '') for i in x])

In [29]:
movies.sample(1)

,movie_id,keywords,title,genres,overview,cast,crew
4065,297621,[woman director],There's Always Woodstock,"[Romance, Comedy, Music]","[When, Neurotic,, struggling, songwriter,, Cat...","[AllisonMiller, JamesWolk, KateySagal]",[RitaMerson]


In [30]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [32]:
new_df = movies[['movie_id', 'title', 'tags']]

In [33]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [34]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

/var/folders/_4/0cj7k8hx41v47wl_c6l982s80000gn/T/ipykernel_8090/1824047427.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [35]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

/var/folders/_4/0cj7k8hx41v47wl_c6l982s80000gn/T/ipykernel_8090/1380776331.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [36]:
new_df.iloc[0]['tags']

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d samworthington zoesaldana sigourneyweaver jamescameron'

### preprocessing is complete

In [37]:
# we want to define 2 similar words such as 'action' and 'actions' as same thin in feature_extraction
# so we use nltk
!pip install nltk 

In [38]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [39]:
#  helper function to stem the text
def stem(text):
    l = []
    for i in text.split():
        l.append(ps.stem(i)) # to stem or to make root word for each word
    return ' '.join(l)

In [40]:
new_df['tags'] = new_df['tags'].apply(stem)

/var/folders/_4/0cj7k8hx41v47wl_c6l982s80000gn/T/ipykernel_8090/3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [41]:
new_df['tags'][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultur clash futur space war space coloni societi space travel futurist romanc space alien tribe alien planet cgi marin soldier battl love affair anti war power relat mind and soul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [42]:
#  do the cv feature extraction steps again

In [43]:
from sklearn.feature_extraction.text import CountVectorizer

In [44]:
# make an object of above class
cv = CountVectorizer(max_features= 5000, stop_words='english')

In [45]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [46]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4806, 5000))

In [47]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

In [48]:
# calculate cosine dist between every pair of movies. cosine dist = k/cosine_similarity

In [49]:
from sklearn.metrics.pairwise import cosine_similarity

In [50]:
similarity = cosine_similarity(vectors)

In [51]:
similarity

array([[1.        , 0.07156563, 0.05249442, ..., 0.04845016, 0.        ,
        0.        ],
       [0.07156563, 1.        , 0.04445542, ..., 0.0410305 , 0.        ,
        0.02198997],
       [0.05249442, 0.04445542, 1.        , ..., 0.02006431, 0.        ,
        0.        ],
       ...,
       [0.04845016, 0.0410305 , 0.02006431, ..., 1.        , 0.03888079,
        0.03969942],
       [0.        , 0.        , 0.        , ..., 0.03888079, 1.        ,
        0.08335142],
       [0.        , 0.02198997, 0.        , ..., 0.03969942, 0.08335142,
        1.        ]], shape=(4806, 4806))

In [52]:
def recommend(movie):
    index = new_df[new_df['title']==movie].index[0]
    distances = enumerate(similarity[index]) # here distance is similarity
    distances = sorted(distances, reverse=True, key = lambda x: x[1])
    for i in distances[1:6]:
        print(new_df.iloc[i[0]]['title'])

In [53]:
recommend('Batman')

Batman
Batman & Robin
Batman Returns
Batman Begins
The Dark Knight Rises


In [54]:
import pickle
pickle.dump(new_df.to_dict(), open('movies.pkl', 'wb'))

In [55]:
new_df['title'].values

array(['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre',
       ..., 'Signed, Sealed, Delivered', 'Shanghai Calling',
       'My Date with Drew'], shape=(4806,), dtype=object)

In [56]:
pickle.dump(similarity, open('similarity.pkl', 'wb'))